# Chapter 0 — Setup

이 노트북을 **한 번만** 실행하면 Ch 1~5 노트북이 즉시 작동합니다.

## 검증 항목
1. `.env`에 필요한 키가 있는지 (4 LLM provider + Opik + Neo4j)
2. Opik 트레이싱이 워크스페이스 `seocho`에 본인 프로젝트를 생성하는지
3. FinDER 데이터셋 다운로드 + 카테고리 분포 확인
4. Neo4j/DozerDB 연결
5. 4-provider 동시 호출이 모두 응답하는지

## 사전 준비 (`.env` 예시)
```dotenv
# --- LLM providers (최소 1개, 권장 4개 모두) ---
OPENAI_API_KEY=sk-...
MOONSHOT_API_KEY=...
DEEPSEEK_API_KEY=...
XAI_API_KEY=...

# --- Opik (워크스페이스 seocho에 초대된 멤버 기준) ---
OPIK_API_KEY=...
OPIK_WORKSPACE=seocho
OPIK_USER=hardy           # ← 본인 식별자. 프로젝트가 teaching-ch{N}-hardy 로 만들어짐

# --- Neo4j / DozerDB ---
NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=...
```

In [ ]:
# === Bootstrap — runs in Colab OR locally ===
# 1) Colab 인 경우 Drive 마운트 후 teaching-resource 폴더로 이동.
#    경로는 본인 Drive 구조에 맞게 수정.
# 2) 자격증명은 Colab Secrets (좌측 🔑) 또는 로컬 .env 에서 로드.
#    절대 평문으로 노트북에 적지 마세요.
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # ↓ 본인 Drive 의 teaching-resource 위치로 바꾸세요.
    CANDIDATES = [
        '/content/drive/MyDrive/26년도_seocho/260425_3주차/teaching-resource',
        '/content/teaching-resource',
    ]
else:
    CANDIDATES = [
        str(Path.cwd()),
        str(Path.cwd() / 'teaching-resource'),
        '/home/hadry/lab/seocho/teaching-resource',
    ]

HERE = next((p for p in CANDIDATES if Path(p).exists() and (Path(p) / '_shared').exists()), None)
if HERE is None:
    raise FileNotFoundError(f'teaching-resource not found in {CANDIDATES}')
os.chdir(HERE)
if HERE not in sys.path:
    sys.path.insert(0, HERE)
print('cwd:', os.getcwd())


In [ ]:
# === Credentials — Colab Secrets 또는 .env 에서 로드 ===
# Colab: 좌측 🔑 → Secrets 에 키 등록 후 notebook access ON.
# Local: teaching-resource/.env 또는 repo 루트의 .env 에 KEY=VALUE 형태로 저장.
def _set(env_key, default=None):
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(env_key)
        except Exception:
            val = None
    if val:
        os.environ[env_key] = val
    elif default is not None and not os.environ.get(env_key):
        os.environ[env_key] = default

# 로컬: dotenv 로 .env 흡수
if not IN_COLAB:
    try:
        from dotenv import load_dotenv
        for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
            if cand.exists():
                load_dotenv(cand, override=False)
                print(f'loaded {cand}')
                break
    except ImportError:
        pass

for key in ['OPENAI_API_KEY','MOONSHOT_API_KEY','DEEPSEEK_API_KEY','XAI_API_KEY',
            'OPIK_API_KEY','NEO4J_URI','NEO4J_PASSWORD']:
    _set(key)
_set('OPIK_WORKSPACE', default='seocho')
_set('OPIK_USER',      default='anonymous')
_set('NEO4J_USER',     default='neo4j')

# 결과 — 키 자체는 출력하지 않고 SET/UNSET 만 표시
for k in ['OPENAI_API_KEY','MOONSHOT_API_KEY','DEEPSEEK_API_KEY','XAI_API_KEY',
          'OPIK_API_KEY','OPIK_WORKSPACE','OPIK_USER','NEO4J_URI','NEO4J_PASSWORD']:
    v = os.environ.get(k, '')
    flag = 'SET' if v else 'UNSET'
    print(f'  {k:18s} = {flag}')


In [27]:
from pathlib import Path
import os, sys

# 노트북 CWD를 teaching-resource/ 로 강제 (어디서 열어도 import 가능)
HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# .env 로딩 (repo 루트 또는 teaching-resource/)
from dotenv import load_dotenv
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False)
        print(f'loaded {cand}')
        break
else:
    print('no .env found — environment vars must already be set')

print('cwd:', Path.cwd())

no .env found — environment vars must already be set
cwd: /content/drive/MyDrive/26년도_seocho/260425_3주차/teaching-resource


## 1. Provider 가용성 점검

4 provider 중 어느 키가 설정됐는지 한눈에 확인. OpenAI는 필수, 나머지는 권장.

In [ ]:
from _shared.providers import providers_overview, available_providers

df = providers_overview()
df

In [ ]:
avail = available_providers()
if not avail.get('openai'):
    print('⚠️  OpenAI 키가 없으면 일부 데모가 skip 됩니다. .env 에 OPENAI_API_KEY 를 추가하세요.')
else:
    configured = [n for n, ok in avail.items() if ok]
    print(f'configured providers: {configured}')

## 2. Opik 트레이싱

- 워크스페이스: `OPIK_WORKSPACE` (default `seocho`)
- 프로젝트: `teaching-ch{N}-{OPIK_USER}` 자동 생성 — 멤버별 분리
- 백엔드: Opik + JSONL(`./traces/chapter_00.jsonl`) 이중
- Opik 키가 없으면 `only_jsonl=True` 로 fallback

참고: <https://www.comet.com/docs/opik/evaluation/evaluate_agents>

In [ ]:
from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik

has_opik_key = bool(os.getenv('OPIK_API_KEY'))
info = setup_opik('00', only_jsonl=not has_opik_key)
info

In [ ]:
if info['degraded']:
    print('Opik degraded:', info['degraded_reasons'])
    print('→ 키/네트워크를 점검해주세요. JSONL 트레이스는 계속 기록됩니다.')
elif 'opik' in info['backends']:
    print('Opik UI:', opik_console_link(info['project']))
else:
    print('JSONL only —', info['jsonl_path'])

## 3. FinDER 데이터셋 로드

ICLR'25 FinDER (5,703 QA, 10-K SEC filings, S&P 500). 8 카테고리.

최초 1회만 HuggingFace에서 다운로드 → `data/finder_corpus.parquet`로 캐싱.

In [ ]:
from _shared.finder_loader import (
    load_finder, by_category, sample_random, sample_per_category,
    category_distribution, FINDER_CATEGORIES,
)

ds = load_finder()
print('total records:', len(ds))
print('fields       :', list(ds.features))

In [ ]:
category_distribution()

### 샘플러 데모

In [ ]:
# (a) 카테고리별 — Risk 카테고리만
risk = by_category('Risk')
print('Risk records:', len(risk))
print('첫 질문    :', risk[0]['question'][:140])

In [ ]:
# (b) 랜덤 — 5건
rs = sample_random(5)
for r in rs:
    print(f"- [{r['category']}] {r['question'][:100]}")

In [ ]:
# (c) 카테고리당 N개씩 균등 — Ch 3~5 평가셋에 자주 쓰임
balanced = sample_per_category(2)
print(f'{len(balanced)} records ({len(FINDER_CATEGORIES)} categories × 2)')
for r in balanced[:8]:
    print(f"  [{r['category']:18s}] {r['question'][:70]}")

## 4. Neo4j / DozerDB 연결

Ch 1~2는 그래프 DB 필수. 미구성이면 메시지만 출력하고 넘어감.

In [ ]:
neo4j_uri = os.getenv('NEO4J_URI')
neo4j_pwd = os.getenv('NEO4J_PASSWORD')
if not (neo4j_uri and neo4j_pwd):
    print('⚠️  NEO4J_URI / NEO4J_PASSWORD 미설정 — Ch 1~2 실습에서 필요합니다.')
else:
    try:
        from neo4j import GraphDatabase
        drv = GraphDatabase.driver(neo4j_uri, auth=(os.getenv('NEO4J_USER', 'neo4j'), neo4j_pwd))
        with drv.session() as s:
            v = s.run('CALL dbms.components() YIELD name, versions RETURN name, versions[0] AS v').data()
        drv.close()
        print('Neo4j/DozerDB 연결 OK →', v)
    except Exception as exc:
        print(f'Neo4j 연결 실패: {type(exc).__name__}: {exc}')

## 5. 4-Provider Smoke Test

같은 짧은 프롬프트를 가용 provider 모두에 보내 응답/토큰/지연을 비교.
이 셀이 통과하면 Ch 1~5의 `compare_providers(...)` 셀이 모두 작동합니다.

In [ ]:
from _shared.providers import compare_providers

prompt = (
    'FinDER 벤치마크의 8개 카테고리 중 "Risk" 카테고리에 어떤 종류의 질문이 들어가는지 한 문장으로 답해주세요.'
)
df = compare_providers(prompt, max_tokens=200)
df

In [ ]:
if df.empty:
    print('⚠️ 아직 호출된 provider 가 없습니다. 최소 1개 키를 .env 에 설정하세요.')
else:
    show = df[['provider', 'model', 'total_tokens', 'latency_ms']].copy()
    show['ms/tok'] = (show['latency_ms'] / show['total_tokens'].clip(lower=1)).round(2)
    print(show.to_string(index=False))

## 6. Teardown

Opik 트레이스를 flush. 다음 챕터부터는 노트북마다 `setup_opik(...)` / `teardown_opik()` 페어를 호출하세요.

In [ ]:
teardown_opik()
print('done. 다음 단계 → chapter-01-knowledge-graph-indexing.ipynb')